# Fine-tuning LoRA — Assistente Médico (Apple Silicon)

Dataset: `assets/sft_positive_conversations.jsonl` (gerado por `export-positive-conversations.ipynb`).

Cada linha é uma chamada LLM (`generate`, `router`, `rewrite`, `guardrail_classify`, `guardrail_regenerate`, …). Linhas **`generate`** já incluem **histórico multi-turno** em `llm_input` (system + turnos user/assistant anteriores + mensagem final com contexto clínico/PCDT).


In [1]:
!uv pip install mlx-tune mlx-lm transformers datasets torch TensorFlow ipywidgets

Using Python 3.11.14 environment at: /Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/.venv
Audited 7 packages in 72ms


# Carregando o modelo base

In [38]:
import os

from mlx_tune import FastLanguageModel, SFTTrainer, SFTConfig
from datasets import load_dataset

import json
from pathlib import Path

In [ ]:
max_seq_length = 8192  # reduza (4096/2048) se o kernel cair por memória

# Treino LoRA — memória (KV cache NÃO reduz RAM no treino; só na inferência)
TRAIN_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
VAL_BATCHES = 0  # 0 = sem val durante treino (evita pico na iteração 1)
CLEAR_CACHE_THRESHOLD_GB = 4.0  # mlx_lm limpa cache Metal entre steps acima disso
FILTER_OVERLONG_EXAMPLES = True  # remove linhas com mais tokens que max_seq_length

# Token HF opcional (export Hugging Face do modelo base)
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="mlx-community/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None,
    token=HF_TOKEN,
)

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

## Carregando adaptadores LoRA

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Rank: 16 is sweet spot for most tasks
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,  # 0 is optimized in Unsloth
    bias="none",
    # Mirrors Unsloth's tutorial syntax exactly. mlx-tune's default is
    # False (faster); pass "unsloth" or True when you need to halve
    # activation memory at the cost of ~2× step time.
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

LoRA configuration set: rank=16, alpha=16, modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], dropout=0


## Formatação para o chat template

Os exemplos vêm do backend (`llm_input` + `llm_output`). **Não** injetamos system prompt externo: cada linha já traz o system da tarefa (`generate`, router, rewrite, guardrail, …).

- **`generate`:** `llm_input` pode ter vários turnos `user`/`assistant` (histórico) + turno final com PCDT; `llm_output` é a resposta alvo.
- **Auxiliares:** prompt curto da tarefa + saída esperada (JSON, consulta reformulada, etc.).

Use `TRAIN_CALL_TYPES` para treinar só `generate` ou todas as tarefas exportadas.

In [20]:
import json
from pathlib import Path

SFT_JSONL_PATH = Path("assets/sft_positive_conversations.jsonl")

# None = todas as linhas exportadas; ou frozenset({"generate"}) para só resposta clínica
TRAIN_CALL_TYPES = None

ROLE_ALIASES = {
    "human": "user",
    "ai": "assistant",
    "assistant": "assistant",
    "user": "user",
    "system": "system",
}


def _coerce_llm_input(llm_input) -> list:
    """Converte llm_input de Dataset/pandas (numpy) para lista de mensagens."""
    if llm_input is None:
        return []
    if hasattr(llm_input, "tolist"):
        llm_input = llm_input.tolist()
    return list(llm_input)


def normalize_messages(llm_input: list[dict]) -> list[dict]:
    """Normaliza roles para o chat template Llama."""
    out: list[dict] = []
    for raw in _coerce_llm_input(llm_input):
        msg = raw.to_dict() if hasattr(raw, "to_dict") else raw
        if not isinstance(msg, dict):
            msg = dict(msg)
        role = ROLE_ALIASES.get((msg.get("role") or "").strip().lower(), msg.get("role"))
        content = msg.get("content")
        if content is None:
            continue
        if isinstance(content, list):
            content = json.dumps(content, ensure_ascii=False)
        out.append({"role": role, "content": str(content)})
    return out


def count_dialog_turns(messages: list[dict]) -> int:
    """Conta mensagens user/assistant no prompt (exclui system)."""
    return sum(1 for m in messages if m.get("role") in ("user", "assistant"))


def messages_to_sft_text(llm_input, llm_output, *, tok) -> str:
    """Histórico completo + resposta alvo, no formato do tokenizer."""
    messages = normalize_messages(llm_input)
    messages.append({"role": "assistant", "content": str(llm_output or "")})
    return tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

## Carregando dataset exportado

In [ ]:
from datasets import load_dataset

if not SFT_JSONL_PATH.is_file():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {SFT_JSONL_PATH}. "
        "Execute export-positive-conversations.ipynb antes."
    )

raw_dataset = load_dataset("json", data_files=str(SFT_JSONL_PATH))
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'patient_id', 'doctor_id', 'message_id', 'call_type', 'sequence', 'model', 'llm_input', 'llm_output'],
        num_rows: 33
    })
})

In [ ]:
import pandas as pd

stats = raw_dataset["train"].to_pandas()
print("Linhas por call_type:\n", stats["call_type"].value_counts(), sep="")

generate_rows = raw_dataset["train"].filter(lambda r: r["call_type"] == "generate")
if len(generate_rows) > 0:
    sample = generate_rows[0]
    turns = count_dialog_turns(normalize_messages(sample["llm_input"]))
    print(f"\nExemplo generate: {turns} mensagens user/assistant no llm_input")
    print(f"conversation_id={sample['conversation_id']}")


def format_sft_row(row):
    messages = normalize_messages(row["llm_input"])
    return {
        "text": messages_to_sft_text(row["llm_input"], row["llm_output"], tok=tokenizer),
        "call_type": row["call_type"],
        "conversation_id": row["conversation_id"],
        "message_id": row["message_id"],
        "dialog_turns": count_dialog_turns(messages),
    }


# Pré-visualização do texto de treino (primeira linha generate)
if len(generate_rows) > 0:
    preview_text = format_sft_row(generate_rows[0])["text"]
    preview_lines = preview_text.splitlines()
    num_lines = len(preview_lines)

    print("\n--- Prévia (início e fim) ---\n")
    if num_lines > 60:
        # Mostra as primeiras 30 e as últimas 30 linhas
        for line in preview_lines[:30]:
            print(line)
        print("\n... [truncado] ...\n")
        for line in preview_lines[-30:]:
            print(line)
    else:
        # Se <= 60, mostra tudo
        print(preview_text)

Linhas por call_type:
call_type
generate              7
router                7
guardrail_classify    7
rewrite               6
rerank                6
Name: count, dtype: int64

Exemplo generate: 1 mensagens user/assistant no llm_input
conversation_id=conv_869c9d985cdc

--- Prévia (início e fim) ---

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 May 2026

Você é um assistente clínico de apoio a médicos no Brasil.
Responda sempre em português do Brasil, de forma objetiva e profissional.
Seja direto: vá ao ponto sem introduções desnecessárias, e use listas apenas quando genuinamente útil.
Nunca invente dados clínicos; quando recorrer ao conhecimento geral sem respaldo de protocolo, sinalize isso claramente.
Nunca utilize placeholders numéricos, textuais genéricos como "[Nome do Médico]" ou inicie com saudações. Vá direto ao resumo ou à resposta.
Só cumprimente se o médico cumprimentar primeiro.

## Contexto por turno
A 

### Transformando o dataset no padrão

In [7]:
train_source = raw_dataset["train"]
if TRAIN_CALL_TYPES is not None:
    train_source = train_source.filter(lambda row: row["call_type"] in TRAIN_CALL_TYPES)

print(f"Exemplos para treino: {len(train_source)}")
if len(train_source) == 0:
    raise ValueError("Nenhum exemplo após filtro TRAIN_CALL_TYPES.")

dataset = train_source.map(format_sft_row)

# Estatísticas de turnos (só generate costuma ter histórico longo)
gen = dataset.filter(lambda r: r["call_type"] == "generate")
if len(gen) > 0:
    turns = gen["dialog_turns"]
    print(
        f"generate: n={len(gen)} turnos user/assistant — "
        f"min={min(turns)} max={max(turns)} média={sum(turns)/len(turns):.1f}"
    )

# Auditoria de comprimento (picos por batch explicam OOM nas iterações 4–6)
def _text_token_len(row) -> int:
    return len(tokenizer.encode(row["text"], add_special_tokens=False))

token_lens = [_text_token_len(dataset[i]) for i in range(len(dataset))]
print(
    f"Tokens/texto: n={len(token_lens)} min={min(token_lens)} "
    f"max={max(token_lens)} média={sum(token_lens)/len(token_lens):.0f}"
)
over = sum(1 for n in token_lens if n > max_seq_length)
if over:
    print(f"  {over} exemplos acima de max_seq_length={max_seq_length}")

if FILTER_OVERLONG_EXAMPLES and over:
    keep = [i for i, n in enumerate(token_lens) if n <= max_seq_length]
    dataset = dataset.select(keep)
    print(f"  Mantidos {len(keep)} exemplos após filtro")

dataset[0]

Exemplos para treino: 33
generate: n=7 turnos user/assistant — min=1 max=5 média=2.1
Tokens/texto: n=33 min=164 max=8380 média=1961
  1 exemplos acima de max_seq_length=8192
  Mantidos 32 exemplos após filtro


{'conversation_id': 'conv_869c9d985cdc',
 'patient_id': 'pt_db65dbac394d',
 'doctor_id': 'dr_1088204c2c24',
 'message_id': 'msg_a26d3c195c99',
 'call_type': 'generate',
 'sequence': -1,
 'model': None,
 'llm_input': [{'role': 'system',
   'content': 'Você é um assistente clínico de apoio a médicos no Brasil.\nResponda sempre em português do Brasil, de forma objetiva e profissional.\nSeja direto: vá ao ponto sem introduções desnecessárias, e use listas apenas quando genuinamente útil.\nNunca invente dados clínicos; quando recorrer ao conhecimento geral sem respaldo de protocolo, sinalize isso claramente.\nNunca utilize placeholders numéricos, textuais genéricos como "[Nome do Médico]" ou inicie com saudações. Vá direto ao resumo ou à resposta.\nSó cumprimente se o médico cumprimentar primeiro.\n\n## Contexto por turno\nA cada turno você pode receber contexto estruturado pelo sistema: dados clínicos do paciente e/ou resultados de busca em PCDTs (Protocolos Clínicos e Diretrizes Terapêuti

# Fine tune em ação

## Configuração do treinamento

In [8]:
training_config = SFTConfig(
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    val_batches=VAL_BATCHES,
    warmup_steps=5,
    max_steps=20,  # Small for demo, use 500+ for real training
    # num_train_epochs=1, # One pass through the dataset
    learning_rate=2e-4,
    logging_steps=1,
    output_dir="outputs",
    optim="adamw_8bit",
    weight_decay=0.01,
    seed=3407,
    lr_scheduler_type="linear",
    max_seq_length=max_seq_length,  # deve bater com mlx-lm (SFTConfig default é 2048)
    grad_checkpoint=True,
    use_native_training=False,  # treino via mlx_lm abaixo, não mlx-tune.train()
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    max_seq_length = max_seq_length,
    dataset_text_field = "text",
    tokenizer=tokenizer,
    # packing=True não é usado pelo mlx_lm.tuner.train (só mlx-tune)
    args=training_config,  # Pass SFTConfig just like in Unsloth!
)

Trainer initialized:
  Output dir: outputs
  Adapter path: outputs/adapters
  Learning rate: 0.0002
  Iterations: 20
  Batch size: 1
  LoRA r=16, alpha=16
  Native training: False
  LR scheduler: linear
  Grad checkpoint: True


### Execução do treinamento

**Memória:** treino faz forward+backward na sequência inteira; **KV cache não ajuda** (só acelera geração token a token na inferência). Picos vêm de `batch_size × comprimento do batch` + validação.

Ajustes neste notebook: `TRAIN_BATCH_SIZE=1`, `VAL_BATCHES=0`, `grad_checkpoint`, `CLEAR_CACHE_THRESHOLD_GB`, filtro de exemplos longos.

Se o kernel cair sem traceback: reduza `max_seq_length`, confira `Peak mem` nos logs da iteração anterior, ou treine via terminal (`python -m mlx_lm.lora ...`) para ver o erro Metal.

In [9]:
# mlx-tune: trainer.train() não repassa max_seq_length corretamente em alguns caminhos
# train_result = trainer.train()

import gc
import types
import traceback

import mlx.core as mx
import mlx.optimizers as optim
from mlx_lm.tuner.datasets import CacheDataset, load_dataset as mlx_load_dataset
from mlx_lm.tuner.trainer import TrainingArgs, train as mlx_train

# Grava outputs/train.jsonl a partir do dataset mapeado
data_dir = trainer._prepare_training_data()

if hasattr(model, "_apply_lora") and not getattr(model, "_lora_applied", False):
    model._apply_lora()

if hasattr(model, "set_adapter_path"):
    model.set_adapter_path(str(trainer.adapter_path))

# Schedule de LR (mesmos hiperparâmetros do SFTConfig)
if training_config.lr_scheduler_type == "linear":
    lr_schedule = optim.linear_schedule(
        init=trainer.learning_rate,
        end=0.0,
        steps=trainer.iters,
    )
elif training_config.lr_scheduler_type == "cosine":
    lr_schedule = optim.cosine_decay(
        init=trainer.learning_rate,
        decay_steps=trainer.iters,
    )
else:
    lr_schedule = trainer.learning_rate

optimizer = optim.AdamW(
    learning_rate=lr_schedule,
    weight_decay=trainer.weight_decay,
)

grad_checkpoint = bool(
    training_config.grad_checkpoint
    or (
        hasattr(model, "lora_config")
        and model.lora_config.get("use_gradient_checkpointing") in (True, "unsloth")
    )
)

clear_cache_bytes = int(CLEAR_CACHE_THRESHOLD_GB * 1e9)

mlx_training_args = TrainingArgs(
    batch_size=trainer.batch_size,
    iters=trainer.iters,
    val_batches=VAL_BATCHES,
    steps_per_report=trainer.logging_steps,
    steps_per_eval=(
        training_config.steps_per_eval
        if training_config.steps_per_eval is not None
        else max(trainer.save_steps, 200)
    ),
    steps_per_save=trainer.save_steps,
    max_seq_length=max_seq_length,
    adapter_file=str(trainer.adapter_path / "adapters.safetensors"),
    grad_checkpoint=grad_checkpoint,
    grad_accumulation_steps=training_config.gradient_accumulation_steps,
    clear_cache_threshold=clear_cache_bytes,
)

print("mlx-lm TrainingArgs:")
print(f"  max_seq_length={mlx_training_args.max_seq_length}")
print(f"  iters={mlx_training_args.iters}, batch_size={mlx_training_args.batch_size}")
print(f"  grad_checkpoint={mlx_training_args.grad_checkpoint}")
print(f"  val_batches={mlx_training_args.val_batches}")
print(f"  clear_cache_threshold={CLEAR_CACHE_THRESHOLD_GB} GB")

dataset_args = types.SimpleNamespace(
    data=data_dir,
    train=True,
    test=False,
    hf_dataset=None,
    mask_prompt=False,
)

train_set, valid_set, _ = mlx_load_dataset(
    args=dataset_args,
    tokenizer=tokenizer,
)
train_set = CacheDataset(train_set)
valid_set = CacheDataset(valid_set)
print(f"Amostras: train={len(train_set)}, valid={len(valid_set)}")

actual_model = model.model if hasattr(model, "model") else model
gc.collect()

val_batches = max(0, int(mlx_training_args.val_batches))
effective_val_set = valid_set if val_batches > 0 else None

print("Iniciando mlx_lm.tuner.train ...")
try:
    mlx_train(
        model=actual_model,
        optimizer=optimizer,
        train_dataset=train_set,
        val_dataset=effective_val_set,
        args=mlx_training_args,
    )
    trainer._save_adapter_config()
    train_result = {"status": "success", "adapter_path": str(trainer.adapter_path)}
except Exception as exc:
    print("Treino falhou com exceção Python:")
    traceback.print_exc()
    raise
finally:
    gc.collect()
    if hasattr(mx, "clear_cache"):
        mx.clear_cache()

train_result

Preparing training data...
  Detected format: text
✓ Prepared 32 training samples
  Saved to: outputs/train.jsonl
✓ Created validation set (copied from train)
Applying LoRA to 28 layers: {'rank': 16, 'scale': 1.0, 'dropout': 0, 'keys': ['mlp.down_proj', 'mlp.gate_proj', 'mlp.up_proj', 'self_attn.k_proj', 'self_attn.o_proj', 'self_attn.q_proj', 'self_attn.v_proj']}
✓ LoRA applied successfully to 28 layers
  Trainable LoRA parameters: 392
mlx-lm TrainingArgs:
  max_seq_length=8192
  iters=20, batch_size=1
  grad_checkpoint=True
  val_batches=0
  clear_cache_threshold=4.0 GB
Amostras: train=32, valid=32
Iniciando mlx_lm.tuner.train ...
Starting training..., iters: 20
Iter 1: Train loss 1.536, Learning Rate 2.000e-04, It/sec 0.056, Tokens/sec 213.633, Trained Tokens 3834, Peak mem 12.170 GB
Iter 2: Train loss 3.846, Learning Rate 2.000e-04, It/sec 1.308, Tokens/sec 215.847, Trained Tokens 3999, Peak mem 12.170 GB
Iter 3: Train loss 1.536, Learning Rate 2.000e-04, It/sec 0.059, Tokens/sec 2

{'status': 'success', 'adapter_path': 'outputs/adapters'}

### Carregar modelo fine-tuned do disco

Use esta célula após reiniciar o kernel (sem repetir treino). Requer `outputs/adapters/adapters.safetensors` e `adapter_config.json` gerados pelo treino.

In [21]:
import os
from pathlib import Path

from mlx_tune import FastLanguageModel

# Mesmo modelo base e pasta de adaptadores do treino
BASE_MODEL_NAME = "mlx-community/Llama-3.2-3B-Instruct"
ADAPTER_DIR = Path("outputs/adapters")

max_seq_length = 8192  # alinhe ao valor usado no treino
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

adapter_weights = ADAPTER_DIR / "adapters.safetensors"
adapter_config = ADAPTER_DIR / "adapter_config.json"
if not adapter_weights.is_file() or not adapter_config.is_file():
    raise FileNotFoundError(
        f"Adaptadores não encontrados em {ADAPTER_DIR.resolve()}. "
        "Execute o treino ou ajuste ADAPTER_DIR."
    )

print(f"Carregando base: {BASE_MODEL_NAME}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=None,
    token=HF_TOKEN,
)

print(f"Carregando LoRA: {ADAPTER_DIR.resolve()}")
model.load_adapter(str(ADAPTER_DIR))

actual_model = model.model if hasattr(model, "model") else model
FastLanguageModel.for_inference(actual_model)

print("Modelo fine-tuned pronto para inferência (actual_model, tokenizer).")

Carregando base: mlx-community/Llama-3.2-3B-Instruct


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Carregando LoRA: /Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/llm/fine-tuning/outputs/adapters
Loading adapters from outputs/adapters...
✓ Adapters loaded successfully
Modelo fine-tuned pronto para inferência (actual_model, tokenizer).


/Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/.venv/lib/python3.11/site-packages/mlx_tune/model.py:390: UserWarning: Model does not support inference mode configuration. Expected MLXModelWrapper, got <class 'mlx_lm.models.llama.Model'>
  warnings.warn(


## Testando

In [39]:
from mlx_lm import generate

FastLanguageModel.for_inference(actual_model)

generate_rows = raw_dataset["train"].filter(lambda r: r["call_type"] == "generate")

# Smoke test: mesmo prefixo de um exemplo generate exportado (multi-turn + PCDT)
if len(generate_rows) > 0:
    sample = generate_rows[0]
    infer_messages = normalize_messages(sample["llm_input"])
else:
    infer_messages = [
        {"role": "user", "content": "O que é herpes zoster?"},
    ]

formatted_prompt = tokenizer.apply_chat_template(
    infer_messages,
    tokenize=False,
    add_generation_prompt=True,
)

generate(
    actual_model,
    tokenizer,
    prompt=formatted_prompt,
    max_tokens=1024,
    verbose=True,
)

/Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/.venv/lib/python3.11/site-packages/mlx_tune/model.py:390: UserWarning: Model does not support inference mode configuration. Expected MLXModelWrapper, got <class 'mlx_lm.models.llama.Model'>
  warnings.warn(


Herpes zoster é uma infecção viral causada pelo vírus da varicela-zoster, que é a mesma causa da varicela. A varicela é uma doença viral que causa erupções cutâneas e febre, geralmente nos primeiros anos de vida. O vírus da varicela-zoster permanece em repouso no nervo periférico após a infecção, e em alguns casos, ele pode se reativar e causar a herpes zoster, que é uma infecção mais grave e dolorosa.

A herpes zoster é caracterizada por uma erupção cutânea de vesículas dolorosas, geralmente em uma única área do corpo, como a região dorsal, a região abdominal ou a região perioral. A erupção pode ser acompanhada de febre, mal-estar e dor.

A herpes zoster é mais comum em pessoas idosas, especialmente em pessoas com mais de 50 anos, e em pessoas com doenças crônicas, como diabetes, HIV/AIDS e doenças autoimunes.

A prevenção da herpes zoster é difícil, mas há algumas medidas que podem ser tomadas:

* Vacinação: A vacinação contra a varicela é recomendada para todos os bebês e crianças m

'Herpes zoster é uma infecção viral causada pelo vírus da varicela-zoster, que é a mesma causa da varicela. A varicela é uma doença viral que causa erupções cutâneas e febre, geralmente nos primeiros anos de vida. O vírus da varicela-zoster permanece em repouso no nervo periférico após a infecção, e em alguns casos, ele pode se reativar e causar a herpes zoster, que é uma infecção mais grave e dolorosa.\n\nA herpes zoster é caracterizada por uma erupção cutânea de vesículas dolorosas, geralmente em uma única área do corpo, como a região dorsal, a região abdominal ou a região perioral. A erupção pode ser acompanhada de febre, mal-estar e dor.\n\nA herpes zoster é mais comum em pessoas idosas, especialmente em pessoas com mais de 50 anos, e em pessoas com doenças crônicas, como diabetes, HIV/AIDS e doenças autoimunes.\n\nA prevenção da herpes zoster é difícil, mas há algumas medidas que podem ser tomadas:\n\n* Vacinação: A vacinação contra a varicela é recomendada para todos os bebês e c

### Prompts do nó `generate` (cópia local)

Espelha `backend/src/assistente_medico_api/graph/nodes/generate.py` — sem import do backend. Os testes abaixo montam o turno final com o mesmo layout de `_build_messages`.

In [27]:
import json

# Cópia de GENERATE_SYSTEM_PROMPT (generate.py)
GENERATE_SYSTEM_PROMPT = """\
Você é um assistente clínico de apoio a médicos no Brasil.
Responda sempre em português do Brasil, de forma objetiva e profissional.
Seja direto: vá ao ponto sem introduções desnecessárias, e use listas apenas quando genuinamente útil.
Nunca invente dados clínicos; quando recorrer ao conhecimento geral sem respaldo de protocolo, sinalize isso claramente.
Nunca utilize placeholders numéricos, textuais genéricos como "[Nome do Médico]" ou inicie com saudações. Vá direto ao resumo ou à resposta.
Só cumprimente se o médico cumprimentar primeiro.

## Contexto por turno
A cada turno você pode receber contexto estruturado pelo sistema: dados clínicos do paciente e/ou resultados de busca em PCDTs (Protocolos Clínicos e Diretrizes Terapêuticas).
O médico não vê esse contexto diretamente — ele só vê suas próprias mensagens e suas respostas.
Use o contexto clínico para personalizar a resposta quando aplicável; não extrapole além do que foi fornecido.
Use pronomes adequados ao gênero do paciente.

## Resultados de busca em PCDTs
Quando resultados forem fornecidos:
- Utilize apenas os trechos relevantes para a pergunta; cite cada um pelo identificador [n].
- Ignore trechos que não contribuam para a resposta.
- Se nenhum resultado for pertinente, informe brevemente que documentos relevantes não foram encontrados — sem listar ou descrever os documentos irrelevantes.

Quando não houver resultados (pergunta conversacional ou de acompanhamento):
- Responda com base no histórico da conversa e no seu conhecimento geral.

## Foco da resposta
Responda diretamente à "Mensagem do médico:", usando o restante do contexto apenas como subsídio.

## Formatação de exames e ações clínicas
Quando apresentar exames pendentes do paciente, use lista markdown:
**Exames pendentes:**
- **[nome do exame]** — solicitado [tempo relativo]

Quando listar ações clínicas sugeridas ou recomendações de protocolo, use lista numerada com tipo entre colchetes:
**Ações sugeridas:**
1. [Exame] descrição
2. [Prescrição] descrição
3. [Observação] descrição
4. [Reavaliação] descrição

Tipos válidos: [Exame], [Prescrição], [Observação], [Reavaliação].\
"""


def build_generate_final_human(
    *,
    patient_context: str = "",
    pcdt_context: str = "(Nenhum trecho recuperado.)",
    doctor_message: str,
) -> str:
    """Mesmo layout do turno final em generate._build_messages."""
    patient_block = (
        f"Contexto clínico do paciente:\n{patient_context.strip()}\n\n"
        if patient_context.strip()
        else ""
    )
    return (
        f"{patient_block}\n\n"
        f"Resultado da busca por trechos PCDT:\n{pcdt_context}\n\n"
        f"Mensagem do médico:\n{doctor_message}\n\n"
    )


def messages_for_generate(
    *,
    doctor_message: str,
    patient_context: str = "",
    pcdt_context: str = "(Nenhum trecho recuperado.)",
    chat_history: list[dict] | None = None,
) -> list[dict]:
    """System + histórico + turno final (espelho de _build_messages)."""
    final_human = build_generate_final_human(
        patient_context=patient_context,
        pcdt_context=pcdt_context,
        doctor_message=doctor_message,
    )
    out: list[dict] = [{"role": "system", "content": GENERATE_SYSTEM_PROMPT}]
    for turn in chat_history or []:
        text = (turn.get("content") or "").strip()
        if not text:
            continue
        role = turn.get("role")
        if role == "user":
            out.append({"role": "user", "content": text})
        elif role == "assistant":
            out.append({"role": "assistant", "content": text})
    out.append({"role": "user", "content": final_human})
    return out


def run_generate(messages: list[dict], *, max_tokens: int = 1024) -> str:
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    return generate(
        actual_model,
        tokenizer,
        prompt=prompt,
        max_tokens=max_tokens,
        verbose=True,
    )


def load_first_training_generate_row() -> dict:
    with SFT_JSONL_PATH.open(encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row.get("call_type") == "generate":
                return row
    raise ValueError("nenhuma linha call_type=generate no JSONL")


def parse_generate_final_user(content: str) -> tuple[str, str, str]:
    """Extrai blocos do turno final exportado para remontar com o template local."""
    marker_patient = "Contexto clínico do paciente:\n"
    marker_pcdt = "Resultado da busca por trechos PCDT:\n"
    marker_doctor = "Mensagem do médico:\n"

    patient_context = ""
    if content.startswith(marker_patient):
        idx_pcdt = content.find(marker_pcdt)
        patient_context = content[len(marker_patient) : idx_pcdt].strip()
        tail = content[idx_pcdt:]
    else:
        tail = content

    idx_doctor = tail.find(marker_doctor)
    if idx_doctor < 0:
        raise ValueError("turno final sem 'Mensagem do médico:'")
    pcdt_context = tail[len(marker_pcdt) : idx_doctor].strip()
    doctor_message = tail[idx_doctor + len(marker_doctor) :].strip()
    return patient_context, pcdt_context, doctor_message

### Teste: paciente fictício, sem trechos PCDT

`patient_id` fictício só para rastreio no notebook (não entra no prompt). PCDT vazio usa o mesmo texto de `generate` quando não há documentos recuperados.

In [26]:
FAKE_PATIENT_ID = "pt-eval-ficticio-8iadt-001"

FAKE_PATIENT_CONTEXT = """\
- Nome: Marina Souza
- Sexo biológico (nascimento): Feminino
- Idade: 58 anos
- Sintomas:
  - fadiga persistente há 3 meses
  - perda de peso não intencional (~4 kg)
- Observações: Não informado
- Medicamentos em uso: Levotiroxina 50 mcg/dia
- CID/diagnóstico: E03.9 — Hipotireoidismo não especificado
- Comorbidades: Hipertensão arterial
- Exames concluídos (últimos 6 meses):
  - TSH: 2,1 mUI/L (há 2 meses)
- Exames pendentes (últimos 6 meses):
  Nenhum
- Ações sugeridas pelo protocolo:
  Nenhuma
"""

messages_no_pcdt = messages_for_generate(
    patient_context=FAKE_PATIENT_CONTEXT,
    pcdt_context="(Nenhum trecho recuperado.)",
    doctor_message=(
        "Quais exames adicionais posso solicitar?"
    ),
)

print(f"patient_id (fictício): {FAKE_PATIENT_ID}")
print("========== geração sem PCDT ==========")
run_generate(messages_no_pcdt)

patient_id (fictício): pt-eval-ficticio-8iadt-001
========== geração sem PCDT ==========
**Resposta do assistente clínico:**

**Ações sugeridas:**

1. **Exame:** TSH e T4 em sangue, em intervalos de 2-3 meses, para monitorar a resposta ao tratamento.
2. **Observação:** Documentar a perda de peso e a fadiga persistente em histórico do paciente.

**Observação adicional:**

A perda de peso e a fadiga persistente são sintomas importantes que devem ser monitorados em conjunto com a TSH e T4. A solicitação de exames adicionais é necessária para avaliar a resposta do paciente ao tratamento e ajustar a terapia hormonal se necessário.
Prompt: 836 tokens, 262.057 tokens-per-sec
Generation: 150 tokens, 27.828 tokens-per-sec
Peak memory: 14.433 GB


'**Resposta do assistente clínico:**\n\n**Ações sugeridas:**\n\n1. **Exame:** TSH e T4 em sangue, em intervalos de 2-3 meses, para monitorar a resposta ao tratamento.\n2. **Observação:** Documentar a perda de peso e a fadiga persistente em histórico do paciente.\n\n**Observação adicional:**\n\nA perda de peso e a fadiga persistente são sintomas importantes que devem ser monitorados em conjunto com a TSH e T4. A solicitação de exames adicionais é necessária para avaliar a resposta do paciente ao tratamento e ajustar a terapia hormonal se necessário.'

### Teste: contexto completo (PCDT + paciente do treino)

Primeira linha `call_type=generate` em `sft_positive_conversations.jsonl`: blocos extraídos do export e remontados com o template local (system + turno final).

In [28]:
train_row = load_first_training_generate_row()
final_user_content = next(
    m["content"]
    for m in reversed(train_row["llm_input"])
    if m.get("role") == "user" and "Mensagem do médico:" in (m.get("content") or "")
)

patient_ctx, pcdt_ctx, doctor_msg = parse_generate_final_user(final_user_content)
messages_full_pcdt = messages_for_generate(
    patient_context=patient_ctx,
    pcdt_context=pcdt_ctx,
    doctor_message=doctor_msg,
)

print(
    f"conversation_id={train_row.get('conversation_id')} "
    f"patient_id={train_row.get('patient_id')}"
)
print(f"Mensagem do médico: {doctor_msg!r}")
print("========== geração com PCDT do treino ==========")
run_generate(messages_full_pcdt)

conversation_id=conv_869c9d985cdc patient_id=pt_db65dbac394d
Mensagem do médico: 'O que é herpes zoster?'
========== geração com PCDT do treino ==========
Herpes zoster é uma infecção viral causada pelo vírus da varicela-zoster, que é a mesma causa da varicela. A varicela é uma doença viral que causa erupções cutâneas e febre, geralmente nos primeiros anos de vida. O vírus da varicela-zoster permanece em repouso no nervo periférico após a infecção, e em alguns casos, ele pode se reativar e causar a herpes zoster, que é uma infecção mais grave e dolorosa.

A herpes zoster é caracterizada por uma erupção cutânea de vesículas dolorosas, geralmente em uma única área do corpo, como a região dorsal, a região abdominal ou a região perioral. A erupção pode ser acompanhada de febre, mal-estar e dor.

A herpes zoster é mais comum em pessoas idosas, especialmente em pessoas com mais de 50 anos, e em pessoas com doenças crônicas, como diabetes, HIV/AIDS e doenças autoimunes.

A prevenção da herpes

'Herpes zoster é uma infecção viral causada pelo vírus da varicela-zoster, que é a mesma causa da varicela. A varicela é uma doença viral que causa erupções cutâneas e febre, geralmente nos primeiros anos de vida. O vírus da varicela-zoster permanece em repouso no nervo periférico após a infecção, e em alguns casos, ele pode se reativar e causar a herpes zoster, que é uma infecção mais grave e dolorosa.\n\nA herpes zoster é caracterizada por uma erupção cutânea de vesículas dolorosas, geralmente em uma única área do corpo, como a região dorsal, a região abdominal ou a região perioral. A erupção pode ser acompanhada de febre, mal-estar e dor.\n\nA herpes zoster é mais comum em pessoas idosas, especialmente em pessoas com mais de 50 anos, e em pessoas com doenças crônicas, como diabetes, HIV/AIDS e doenças autoimunes.\n\nA prevenção da herpes zoster é difícil, mas há algumas medidas que podem ser tomadas:\n\n* Vacinação: A vacinação contra a varicela é recomendada para todos os bebês e c

### Comparando geração fine-tuned com `llm_output` exportado

Avaliação qualitativa em exemplos `generate`: o modelo vê o mesmo `llm_input` (histórico + contexto) e gera uma nova resposta para comparar com o `llm_output` de referência.

In [29]:
import pandas as pd

MAX_EVAL_EXAMPLES = 3  # ajuste conforme tempo disponível

eval_source = raw_dataset["train"].filter(lambda r: r["call_type"] == "generate")
n_eval = min(MAX_EVAL_EXAMPLES, len(eval_source))
eval_subset = eval_source.select(range(n_eval))
df_eval = eval_subset.to_pandas()


def generate_from_exported_prompt(row) -> str:
    """Gera resposta a partir do llm_input exportado (histórico + contexto)."""
    messages = normalize_messages(row["llm_input"])
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    return generate(
        model.model,
        tokenizer,
        prompt=formatted_prompt,
        max_tokens=1024,
        verbose=True,
    )


df_eval["resposta_fine_tuned"] = df_eval.apply(generate_from_exported_prompt, axis=1)
df_eval["dialog_turns"] = df_eval["llm_input"].apply(
    lambda inp: count_dialog_turns(normalize_messages(inp))
)

Herpes zoster é uma infecção viral causada pelo vírus da varicela-zoster, que é a mesma causa da varicela. A varicela é uma doença viral que causa erupções cutâneas e febre, geralmente nos primeiros anos de vida. O vírus da varicela-zoster permanece em repouso no nervo periférico após a infecção, e em alguns casos, ele pode se reativar e causar a herpes zoster, que é uma infecção mais grave e dolorosa.

A herpes zoster é caracterizada por uma erupção cutânea de vesículas dolorosas, geralmente em uma única área do corpo, como a região dorsal, a região abdominal ou a região perioral. A erupção pode ser acompanhada de febre, mal-estar e dor.

A herpes zoster é mais comum em pessoas idosas, especialmente em pessoas com mais de 50 anos, e em pessoas com doenças crônicas, como diabetes, HIV/AIDS e doenças autoimunes.

A prevenção da herpes zoster é difícil, mas há algumas medidas que podem ser tomadas:

* Vacinação: A vacinação contra a varicela é recomendada para todos os bebês e crianças m

In [37]:
try:
    from IPython.display import display
except ImportError:
    display = print  # noqa: A001

import re

def extract_pergunta(llm_input_tail):
    # Extrai o texto após "Mensagem do médico:" (ou retorna "")
    marker = "Mensagem do médico:"
    idx = llm_input_tail.find(marker)
    if idx == -1:
        return ""
    after = llm_input_tail[idx + len(marker):]
    # Remove espaços/quebras iniciais
    pergunta = after.strip().split("\n")[0].strip()
    # Remove \n\n'}] do fim (se houver)
    pergunta = re.sub(r"\\n+|'\}\]$", '', pergunta)
    return pergunta

df_eval["pergunta"] = df_eval["llm_input_tail"].apply(extract_pergunta)

cols = [
    "conversation_id",
    "dialog_turns",
    "pergunta",
    "llm_output",
    "resposta_fine_tuned",
]
with pd.option_context("display.max_colwidth", 120, "display.max_rows", None):
    display(df_eval[cols])

,conversation_id,dialog_turns,pergunta,llm_output,resposta_fine_tuned
0,conv_869c9d985cdc,1,O que é herpes zoster?,Herpes zoster é a manifestação de reativação do vírus Varicela Zoster (VZV).\n\nO VZV é o mesmo vírus responsável pe...,"Herpes zoster é uma infecção viral causada pelo vírus da varicela-zoster, que é a mesma causa da varicela. A varicel..."
1,conv_89707a9475dc,1,O que é a síndrome de Beckwith-Wiedemann?,A Síndrome de Beckwith-Wiedemann (BWS) é uma síndrome genética rara caracterizada pelo crescimento excessivo (macros...,"Rebeca Martins, 24 anos, apresenta uma úlcera indolor no pênis que desapareceu após 10 dias sem tratamento, seguida ..."
2,conv_ad0363899bd8,5,Que dose e esquema seguir para penicilina?,O tratamento de escolha para Sífilis Secundária em adultos é a Benzilpenicilina Benzatina.\n\n**Esquema Terapêutico:...,A dose e o esquema para penicilina benzatina devem ser seguidos do protocolo local para sífilis secundária.\n\n**Dos...


## Dependências (setup único)

Ferramentas usadas neste projeto para exportar o LoRA treinado em MLX → GGUF quantizado → Ollama.

| Ferramenta | Uso |
|------------|-----|
| **cmake** | Compilar binários do llama.cpp (`llama-quantize`, etc.) |
| **llama.cpp** | `convert_hf_to_gguf.py` + `llama-quantize` (passos 3–4 da exportação) |
| **mlx-lm** | `mlx_lm.fuse` — fundir base + adaptadores (já no venv do projeto) |
| **Ollama** | Importar o GGUF via `Modelfile` e rodar localmente |

**Disco:** reserve vários GB em `exports/` (modelo fundido fp16 + GGUF f16 + Q4_K_M).

### 1. llama.cpp (ferramenta compartilhada em `~/tools`)

Execute uma vez no terminal (como você já fez na sua máquina):

```bash
brew install cmake

mkdir -p ~/tools
git clone https://github.com/ggml-org/llama.cpp ~/tools/llama.cpp
cd ~/tools/llama.cpp
uv venv --python=python3.11 .venv
source .venv/bin/activate
cmake -B build && cmake --build build --config Release
uv add pip
pip install -r requirements.txt

```

Confirme que existem:

- `~/tools/llama.cpp/convert_hf_to_gguf.py`
- `~/tools/llama.cpp/build/bin/llama-quantize`

### 2. Ollama

```bash
brew install ollama
# ou https://ollama.com/download
```

### 3. Este notebook

Use o venv do repositório (`llm/` → `pip install -e .`) para `mlx_lm.fuse`. A célula de exportação abaixo chama os scripts do llama.cpp em `~/tools/llama.cpp`.


## Salvando o modelo para Ollama (via llama.cpp)

O export direto `save_pretrained_gguf` do mlx-tune costuma falhar (base quantizada, disco, etc.). O fluxo abaixo segue o pipeline **MLX fuse → llama.cpp GGUF f16 → quantize Q4_K_M → Modelfile**.

### Pipeline

| Passo | Ferramenta | Saída |
|-------|------------|--------|
| 1 | `mlx_lm.fuse` + `--dequantize` | `exports/<tag>/fused_model/` (HF/MLX fundido) |
| 2 | `convert_hf_to_gguf.py` | `<tag>_f16.gguf` |
| 3 | `llama-quantize` | `<tag>_Q4_K_M.gguf` (usado no Ollama) |
| 4 | Notebook | `Modelfile` (não é gerado pelo llama.cpp) |

**Tag sugerida (data de hoje):** `assistente-medico-YYYY-MM-DD` — variável `OLLAMA_MODEL_TAG` na célula abaixo.

**Base:** `mlx-community/Llama-3.2-3B-Instruct` (mesmo do treino; adaptadores em `outputs/adapters`).

### Carregar no Ollama (após a célula de exportação)

Substitua `YYYY-MM-DD` pela data em `OLLAMA_MODEL_TAG`:

```bash
cd llm/fine-tuning/exports/assistente-medico-YYYY-MM-DD
ollama create assistente-medico-YYYY-MM-DD -f Modelfile
ollama run assistente-medico-YYYY-MM-DD
```

Teste:

```bash
ollama run assistente-medico-YYYY-MM-DD "Resuma em 3 linhas o que é herpes zoster."
```

`ollama list` · remover: `ollama rm assistente-medico-YYYY-MM-DD`

> O app em produção pode enviar o system prompt completo por requisição; o `SYSTEM` do Modelfile é um padrão para testes locais.


In [1]:
import subprocess
import sys
from datetime import date
from pathlib import Path

# --- Ajuste aqui ---
EXPORT_BASE_MODEL = "mlx-community/Llama-3.2-3B-Instruct"
ADAPTER_DIR = Path("outputs/adapters")
LLAMA_CPP_ROOT = Path.home() / "tools" / "llama.cpp"
LLAMA_QUANTIZE_BIN = LLAMA_CPP_ROOT / "build" / "bin" / "llama-quantize"
CONVERT_SCRIPT = LLAMA_CPP_ROOT / "convert_hf_to_gguf.py"
LLAMA_CPP_PYTHON = LLAMA_CPP_ROOT / ".venv" / "bin" / "python"

OLLAMA_MODEL_TAG = f"assistente-medico-{date.today().isoformat()}"
# OLLAMA_MODEL_TAG = "assistente-medico-2026-05-26"

OLLAMA_EXPORT_DIR = Path("exports") / OLLAMA_MODEL_TAG
FUSED_DIR = OLLAMA_EXPORT_DIR / "fused_model"
GGUF_F16_PATH = OLLAMA_EXPORT_DIR / f"{OLLAMA_MODEL_TAG}_f16.gguf"
GGUF_Q4_PATH = OLLAMA_EXPORT_DIR / f"{OLLAMA_MODEL_TAG}_Q4_K_M.gguf"
MODelfile_PATH = OLLAMA_EXPORT_DIR / "Modelfile"

OLLAMA_SYSTEM_PROMPT = (
    "Você é um assistente clínico de apoio a médicos no Brasil.\n"
    "Responda sempre em português do Brasil, de forma objetiva e profissional.\n"
    "Nunca invente dados clínicos; quando usar conhecimento geral sem protocolo, sinalize isso.\n"
    "Responda diretamente à pergunta do médico."
)


def run_cmd(cmd: list[str], *, cwd: Path | None = None) -> None:
    """Executa comando e propaga erro com log legível."""
    print("$", " ".join(str(x) for x in cmd))
    subprocess.run(cmd, check=True, cwd=str(cwd) if cwd else None)


def require_path(path: Path, hint: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"{hint}\n  Caminho: {path}")


# --- Pré-requisitos ---
require_path(ADAPTER_DIR / "adapters.safetensors", "Adaptadores LoRA não encontrados. Treine antes.")
require_path(ADAPTER_DIR / "adapter_config.json", "adapter_config.json ausente em outputs/adapters.")
require_path(CONVERT_SCRIPT, "convert_hf_to_gguf.py — veja seção Dependências (clone llama.cpp).")
require_path(LLAMA_QUANTIZE_BIN, "llama-quantize — compile llama.cpp (cmake -B build).")

LLAMA_CPP_VENV_ACTIVATE = LLAMA_CPP_ROOT / ".venv" / "bin" / "activate"
require_path(LLAMA_CPP_VENV_ACTIVATE, "Ative o venv do llama.cpp (veja seção Dependências).")


def run_bash(cmd: str, *, cwd: Path | None = None) -> None:
    """Executa um comando bash com `source` do venv do llama.cpp."""
    full = f"source {LLAMA_CPP_VENV_ACTIVATE} && {cmd}"
    print("$ bash -lc", full)
    subprocess.run(
        ["bash", "-lc", full],
        check=True,
        cwd=str(cwd) if cwd else None,
    )


# convert_hf_to_gguf.py depende de PyTorch; valide que o venv do llama.cpp tem `torch`.
try:
    run_bash("python -c 'import torch'", cwd=LLAMA_CPP_ROOT)
except subprocess.CalledProcessError as exc:
    raise RuntimeError(
        "O venv do llama.cpp não tem `torch`. Instale DENTRO dele:\n"
        "  cd ~/tools/llama.cpp && source .venv/bin/activate && pip install torch\n"
        "Depois reexecute esta célula."
    ) from exc

OLLAMA_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Tag Ollama: {OLLAMA_MODEL_TAG}")
print(f"Pasta de export: {OLLAMA_EXPORT_DIR.resolve()}\n")

# Passo 1 — Fundir base + LoRA (fp16/dequantized)
print("=== Passo 1/3: mlx_lm.fuse ===")
run_cmd(
    [
        sys.executable,
        "-m",
        "mlx_lm.fuse",
        "--model",
        EXPORT_BASE_MODEL,
        "--adapter-path",
        str(ADAPTER_DIR.resolve()),
        "--save-path",
        str(FUSED_DIR.resolve()),
        "--dequantize",
    ],
)

require_path(FUSED_DIR, "Diretório fused_model não criado pelo fuse.")

# Passo 2 — HF fundido → GGUF f16
print("\n=== Passo 2/3: convert_hf_to_gguf (f16) ===")
run_bash(
    " ".join(
        [
            f"python {CONVERT_SCRIPT}",
            f"{FUSED_DIR.resolve()}",
            "--outtype f16",
            f"--outfile {GGUF_F16_PATH.resolve()}",
        ]
    ),
    cwd=LLAMA_CPP_ROOT,
)

require_path(GGUF_F16_PATH, "GGUF f16 não gerado.")

# Passo 3 — Quantizar para Ollama
print("\n=== Passo 3/3: llama-quantize Q4_K_M ===")
run_cmd(
    [
        str(LLAMA_QUANTIZE_BIN),
        str(GGUF_F16_PATH.resolve()),
        str(GGUF_Q4_PATH.resolve()),
        "Q4_K_M",
    ],
)

require_path(GGUF_Q4_PATH, "GGUF Q4_K_M não gerado.")

# Modelfile (Ollama)
modelfile_text = f"""# Gerado em {date.today().isoformat()} — assistente médico (SFT)
FROM ./{GGUF_Q4_PATH.name}

PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER num_ctx 8192
PARAMETER stop "<|eot_id|>"

SYSTEM \"\"\"{OLLAMA_SYSTEM_PROMPT}\"\"\"
"""

MODelfile_PATH.write_text(modelfile_text, encoding="utf-8")

print("\n=== Exportação concluída ===")
print(f"  GGUF f16: {GGUF_F16_PATH.resolve()}")
print(f"  GGUF Q4_K_M: {GGUF_Q4_PATH.resolve()}")
print(f"  Modelfile: {MODelfile_PATH.resolve()}")
print("\nTerminal:")
print(f"  cd {OLLAMA_EXPORT_DIR.resolve()}")
print(f"  ollama create {OLLAMA_MODEL_TAG} -f Modelfile")
print(f"  ollama run {OLLAMA_MODEL_TAG}")


$ bash -lc source /Users/leanderseefeld/tools/llama.cpp/.venv/bin/activate && python -c 'import torch'
Tag Ollama: assistente-medico-2026-05-26
Pasta de export: /Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/llm/fine-tuning/exports/assistente-medico-2026-05-26

=== Passo 1/3: mlx_lm.fuse ===
$ /Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/.venv/bin/python -m mlx_lm.fuse --model mlx-community/Llama-3.2-3B-Instruct --adapter-path /Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/llm/fine-tuning/outputs/adapters --save-path /Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/llm/fine-tuning/exports/assistente-medico-2026-05-26/fused_model --dequantize
Calling `python -m mlx_lm.fuse...` directly is deprecated. Use `mlx_lm.fuse...` or `python -m mlx_lm fuse ...` instead.
Loading pretrained model


Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 73035.14it/s]


Dequantizing model



=== Passo 2/3: convert_hf_to_gguf (f16) ===
$ bash -lc source /Users/leanderseefeld/tools/llama.cpp/.venv/bin/activate && python /Users/leanderseefeld/tools/llama.cpp/convert_hf_to_gguf.py /Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/llm/fine-tuning/exports/assistente-medico-2026-05-26/fused_model --outtype f16 --outfile /Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/llm/fine-tuning/exports/assistente-medico-2026-05-26/assistente-medico-2026-05-26_f16.gguf


INFO:hf-to-gguf:Loading model: fused_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {3072, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {3072, 8192}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,   


=== Passo 3/3: llama-quantize Q4_K_M ===
$ /Users/leanderseefeld/tools/llama.cpp/build/bin/llama-quantize /Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/llm/fine-tuning/exports/assistente-medico-2026-05-26/assistente-medico-2026-05-26_f16.gguf /Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/llm/fine-tuning/exports/assistente-medico-2026-05-26/assistente-medico-2026-05-26_Q4_K_M.gguf Q4_K_M


ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 7.518 sec
ggml_metal_rsets_init: creating a residency set collection (keep_alive = 180 s)
ggml_metal_device_init: GPU name:   MTL0 (Apple M4 Pro)
ggml_metal_device_init: GPU family: MTLGPUFamilyApple9  (1009)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal4  (5002)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSetSize  = 19069.67 MB
llama_print_build_info: build = 9348 (5190c2ea8)
llama_print_


llama_quantize: quantize time = 15237.64 ms
llama_quantize:    total time = 15237.64 ms

=== Exportação concluída ===
  GGUF f16: /Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/llm/fine-tuning/exports/assistente-medico-2026-05-26/assistente-medico-2026-05-26_f16.gguf
  GGUF Q4_K_M: /Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/llm/fine-tuning/exports/assistente-medico-2026-05-26/assistente-medico-2026-05-26_Q4_K_M.gguf
  Modelfile: /Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/llm/fine-tuning/exports/assistente-medico-2026-05-26/Modelfile

Terminal:
  cd /Users/leanderseefeld/workspaces/fiap/8iadt-tc-fase3-assistente-medico/llm/fine-tuning/exports/assistente-medico-2026-05-26
  ollama create assistente-medico-2026-05-26 -f Modelfile
  ollama run assistente-medico-2026-05-26
